# Gated Fusion Ablation Study

텍스트만 사용, 텍스트 + 언어적 특성, 텍스트 + 가독성 특성, 텍스트 + 감정 특성, 텍스트 + 행동 특성 절제연구를 한 번에 실행하는 노트북입니다.


## Import

In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from transformers import BertTokenizerFast, TFBertModel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


## GPU

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print(gpus)

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)


## Data Load

In [ ]:
parquet_path = Path("final_data.parquet")

df = pd.read_parquet(parquet_path)

print(df.shape)
print(df["fake"].value_counts().to_dict())
df.head()


## Feature Columns

In [ ]:
basic_cols = [f"basic_{i}" for i in range(13)]
read_cols  = [f"read_{i}" for i in range(6)]
senti_cols = [f"senti_{i}" for i in range(7)]
behav_cols = [f"behav_{i}" for i in range(9)]

df[basic_cols] = pd.DataFrame(df["basic_linguistic_list"].tolist(), index=df.index)
df[read_cols]  = pd.DataFrame(df["readability_list"].tolist(), index=df.index)
df[senti_cols] = pd.DataFrame(df["sentiment_list"].tolist(), index=df.index)
df[behav_cols] = pd.DataFrame(df["behavioral_list"].tolist(), index=df.index)

EXPERIMENTS = {
    "text_only": [],
    "text_basic": basic_cols,
    "text_readability": read_cols,
    "text_sentiment": senti_cols,
    "text_behavioral": behav_cols,
}

for name, cols in EXPERIMENTS.items():
    print(f"{name:16s}: {len(cols)} features")


## Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["fake"]),
    df["fake"].astype("float32"),
    test_size=0.2,
    stratify=df["fake"],
    random_state=SEED,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.125,
    stratify=y_train,
    random_state=SEED,
)

print(len(X_train), len(X_val), len(X_test))


## Tokenize

In [ ]:
MAX_LEN = 256
TEXT_COL = "review_text"

# 빠른 테스트가 필요하면 MAX_LEN을 128로 낮춰도 됩니다.
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

train_enc = tokenizer(
    list(X_train[TEXT_COL]),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="np",
)

val_enc = tokenizer(
    list(X_val[TEXT_COL]),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="np",
)

test_enc = tokenizer(
    list(X_test[TEXT_COL]),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="np",
)


## Model Builder

In [ ]:
def masked_mean_pooling(inputs):
    x, mask = inputs
    mask = tf.cast(mask, tf.float32)
    mask = tf.expand_dims(mask, axis=-1)
    denom = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-6)
    return tf.reduce_sum(x * mask, axis=1) / denom


def build_model(feature_dim, max_len=MAX_LEN, bert_name="bert-base-uncased"):
    bert = TFBertModel.from_pretrained(bert_name)
    bert.trainable = False

    ids = tf.keras.Input((max_len,), dtype=tf.int32, name="input_ids")
    mask = tf.keras.Input((max_len,), dtype=tf.int32, name="attention_mask")

    text = bert(ids, attention_mask=mask).last_hidden_state
    text = tf.keras.layers.Dense(128, activation="gelu")(text)
    text_pool = tf.keras.layers.Lambda(masked_mean_pooling)([text, mask])

    if feature_dim == 0:
        x = text_pool
        model_inputs = {
            "input_ids": ids,
            "attention_mask": mask,
        }
    else:
        features = tf.keras.Input((feature_dim,), dtype=tf.float32, name="features")

        feature_vec = tf.keras.layers.Dense(128, activation="gelu")(features)
        feature_vec = tf.keras.layers.Dense(128, activation="gelu")(feature_vec)

        gate_input = tf.keras.layers.Concatenate()([text_pool, feature_vec])
        gate = tf.keras.layers.Dense(128, activation="sigmoid")(gate_input)

        text_part = tf.keras.layers.Multiply()([gate, text_pool])
        inv_gate = tf.keras.layers.Lambda(lambda g: 1.0 - g)(gate)
        feat_part = tf.keras.layers.Multiply()([inv_gate, feature_vec])
        x = tf.keras.layers.Add()([text_part, feat_part])

        model_inputs = {
            "input_ids": ids,
            "attention_mask": mask,
            "features": features,
        }

    x = tf.keras.layers.Dense(256, activation="gelu")(x)
    x = tf.keras.layers.Dense(64, activation="gelu")(x)
    out = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs=model_inputs, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


## Experiment Runner

In [ ]:
EPOCHS = 20
BATCH_SIZE = 32
PRED_BATCH_SIZE = 64
PATIENCE = 5


def make_inputs(feature_cols):
    train_inputs = {
        "input_ids": train_enc["input_ids"],
        "attention_mask": train_enc["attention_mask"],
    }
    val_inputs = {
        "input_ids": val_enc["input_ids"],
        "attention_mask": val_enc["attention_mask"],
    }
    test_inputs = {
        "input_ids": test_enc["input_ids"],
        "attention_mask": test_enc["attention_mask"],
    }

    if len(feature_cols) > 0:
        scaler = StandardScaler()
        train_inputs["features"] = scaler.fit_transform(X_train[feature_cols]).astype("float32")
        val_inputs["features"] = scaler.transform(X_val[feature_cols]).astype("float32")
        test_inputs["features"] = scaler.transform(X_test[feature_cols]).astype("float32")

    return train_inputs, val_inputs, test_inputs


def run_experiment(experiment_name, feature_cols):
    print("=" * 80)
    print(f"Experiment: {experiment_name} | feature_dim={len(feature_cols)}")

    tf.keras.backend.clear_session()
    gc.collect()
    tf.random.set_seed(SEED)

    train_inputs, val_inputs, test_inputs = make_inputs(feature_cols)
    model = build_model(feature_dim=len(feature_cols))

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
    )

    history = model.fit(
        train_inputs,
        y_train,
        validation_data=(val_inputs, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=1,
    )

    y_prob = model.predict(test_inputs, batch_size=PRED_BATCH_SIZE)
    y_pred = (y_prob > 0.5).astype(int)

    result = {
        "experiment": experiment_name,
        "feature_dim": len(feature_cols),
        "best_val_loss": min(history.history["val_loss"]),
        "best_val_accuracy": max(history.history["val_accuracy"]),
        "test_acc": accuracy_score(y_test, y_pred),
        "test_precision": precision_score(y_test, y_pred),
        "test_recall": recall_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "epochs_ran": len(history.history["loss"]),
    }

    del model, train_inputs, val_inputs, test_inputs
    tf.keras.backend.clear_session()
    gc.collect()

    return result


## Run All Ablation Experiments

In [ ]:
results = []

for experiment_name, feature_cols in EXPERIMENTS.items():
    result = run_experiment(experiment_name, feature_cols)
    results.append(result)

results_df = pd.DataFrame(results)
results_df


## Save Results

In [ ]:
results_path = Path("gated_fusion_ablation_results.csv")
results_df.to_csv(results_path, index=False)
print(results_path.resolve())
results_df.sort_values("test_f1", ascending=False)
